# FMP Macro

Read-first examples for FMP-backed macroeconomic series, treasury data, yield curve history, calendars, and risk premium snapshots.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

In [ ]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Available Macro Sections

In [ ]:
macro_route_libraries = {
    MACRO_ECONOMIC_SECTION: MACRO_ECONOMIC_LIBRARY,
    MACRO_TREASURY_SECTION: MACRO_TREASURY_LIBRARY,
    MACRO_YIELD_CURVE_SECTION: MACRO_YIELD_CURVE_LIBRARY,
    MACRO_CALENDAR_SECTION: MACRO_CALENDAR_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION: MACRO_RISK_PREMIUM_LIBRARY,
}
route_table(macro_route_libraries)

In [ ]:
pd.Series(DEFAULT_ECONOMIC_SERIES, name="default_economic_series")

## Refresh Controls

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_macro(
        economic_series=["GDP", "CPI"],
        include_treasury_rates=True,
        include_yield_curve=True,
        include_calendar=True,
        include_risk_premium=True,
        provider=PROVIDER,
    )

## Storage Coverage

In [ ]:
macro_states = [
    state("GDP", MACRO_ECONOMIC_SECTION),
    state("CPI", MACRO_ECONOMIC_SECTION),
    state(TREASURY_BUNDLE_SYMBOL, MACRO_TREASURY_SECTION),
    state(YIELD_CURVE_BUNDLE_SYMBOL, MACRO_YIELD_CURVE_SECTION),
    state(MACRO_CALENDAR_SYMBOL, MACRO_CALENDAR_SECTION),
    state(RISK_PREMIUM_SYMBOL, MACRO_RISK_PREMIUM_SECTION),
]
pd.DataFrame([row for row in macro_states if row is not None])

## Read Examples

In [ ]:
preview(warehouse.read_macro_panel(["GDP", "CPI"], provider=PROVIDER))

In [ ]:
preview(warehouse.macro.read_calendar(provider=PROVIDER))

In [ ]:
warehouse.macro.read_risk_premium(provider=PROVIDER)

<!-- quant-trader-eda -->
## Quant Trader EDA

Macro checks: growth/inflation changes, equity sensitivity to macro series, and yield-curve regime diagnostics.

In [ ]:
macro_panel = warehouse.read_macro_panel(["GDP", "CPI", "unemploymentRate", "federalFunds"], provider=PROVIDER)
if macro_panel.empty:
    macro_changes = pd.DataFrame()
else:
    macro_numeric = macro_panel.apply(pd.to_numeric, errors="coerce")
    macro_changes = pd.concat(
        {
            "level": macro_numeric,
            "change_1": macro_numeric.diff(),
            "pct_change_1": macro_numeric.pct_change(fill_method=None),
        },
        axis=1,
    )
preview(macro_changes, rows=10)

In [ ]:
spy_prices = warehouse.etf.read_prices("SPY", provider=PROVIDER)
macro_panel = warehouse.read_macro_panel(["GDP", "CPI", "unemploymentRate", "federalFunds"], provider=PROVIDER)
if spy_prices.empty or macro_panel.empty:
    macro_equity_corr = pd.DataFrame()
else:
    monthly_spy = spy_prices["close"].resample("ME").last().pct_change().rename("SPY_monthly_return")
    monthly_macro = macro_panel.apply(pd.to_numeric, errors="coerce").resample("ME").last().ffill().diff()
    joined = pd.concat([monthly_spy, monthly_macro], axis=1).dropna()
    macro_equity_corr = joined.corr()[["SPY_monthly_return"]].drop(index="SPY_monthly_return", errors="ignore")
macro_equity_corr

In [ ]:
treasury_codes = warehouse.macro.list_treasury_series_codes(provider=PROVIDER)
treasury_codes[:20]

In [ ]:
if not treasury_codes:
    pd.DataFrame()
else:
    treasury_panel = pd.concat(
        {code: warehouse.macro.read_series(code, provider=PROVIDER) for code in treasury_codes[:20]},
        axis=1,
    ).dropna(how="all")
    preview(treasury_panel, rows=10)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

Macro data is usually useful when converted into regimes. These cells compare equity returns across inflation, policy-rate, unemployment, and yield-curve states.

In [ ]:
spy_prices = warehouse.etf.read_prices("SPY", provider=PROVIDER)
macro_panel = warehouse.read_macro_panel(["CPI", "unemploymentRate", "federalFunds"], provider=PROVIDER)
if spy_prices.empty or macro_panel.empty:
    pd.DataFrame()
else:
    spy_monthly = spy_prices["close"].resample("ME").last().pct_change().rename("spy_return_fwd_1m").shift(-1)
    macro_monthly = macro_panel.apply(pd.to_numeric, errors="coerce").resample("ME").last().ffill()
    regimes = pd.DataFrame(index=macro_monthly.index)
    if "CPI" in macro_monthly.columns:
        regimes["cpi_rising"] = macro_monthly["CPI"].diff() > 0
    if "unemploymentRate" in macro_monthly.columns:
        regimes["unemployment_rising"] = macro_monthly["unemploymentRate"].diff() > 0
    if "federalFunds" in macro_monthly.columns:
        regimes["fed_funds_rising"] = macro_monthly["federalFunds"].diff() > 0
    joined = pd.concat([spy_monthly, regimes], axis=1).dropna()
    rows = []
    for regime_column in regimes.columns:
        grouped = joined.groupby(regime_column)["spy_return_fwd_1m"].agg(["count", "mean", "std"])
        grouped["annualized_mean"] = grouped["mean"] * 12
        grouped["annualized_vol"] = grouped["std"] * 12 ** 0.5
        grouped["regime"] = regime_column
        rows.append(grouped.reset_index().rename(columns={regime_column: "regime_value"}))
    pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [ ]:
treasury_codes = warehouse.macro.list_treasury_series_codes(provider=PROVIDER)
if not treasury_codes:
    pd.DataFrame()
else:
    treasury_panel = pd.concat(
        {code: warehouse.macro.read_series(code, provider=PROVIDER) for code in treasury_codes},
        axis=1,
    ).dropna(how="all")
    candidate_10y = next((column for column in treasury_panel.columns if "10" in column), None)
    candidate_2y = next((column for column in treasury_panel.columns if "2" in column), None)
    if candidate_10y is None or candidate_2y is None:
        preview(treasury_panel, rows=10)
    else:
        curve = (treasury_panel[candidate_10y] - treasury_panel[candidate_2y]).rename("curve_10y_2y")
        curve_state = pd.DataFrame({
            "curve_10y_2y": curve,
            "curve_change_21d": curve.diff(21),
            "inverted": curve < 0,
        })
        preview(curve_state, rows=15)

<!-- output-analysis -->
## Analysis Notes From This Run

In the current stored macro sample, the simple one-month-forward `SPY` regime split does not show a large penalty from rising CPI, rising unemployment, or rising Fed funds. Annualized forward `SPY` returns were similar across the simple rising/falling policy-rate split, and even slightly higher in the rising CPI/unemployment buckets in this crude diagnostic.

This does not mean macro is irrelevant. It means these raw level-change flags are too blunt to trade directly. For money-making research, the next step is to test surprises, z-scores, yield-curve slope/inversion, inflation acceleration, and interactions with market trend/volatility rather than using simple up/down macro changes alone.